In [1]:
import numpy as np
import tensorflow as tf

# Steps of Training a ML Model

1. Pass data through model to get predicted values.
2. Calculate loss with predicted values and ground truth.
3. Perform backpropagation with loss to get gradients of parameters.
4. Update parameters with gradients using an optimizer.

# Parameters

- Concept: 
Parameters are the trainable variables inside a model—the weights ($w$) and biases ($b$). They are learned automatically from data during training.
- Math Representation: 
We group all parameters into a single vector or set denoted as $\theta$ (theta). $$\theta = [w_1, w_2, \dots, w_n, b]^T$$

- Intuition: 
Think of a neural network as a giant audio mixing board with millions of knobs. The parameters $\theta$ are the current dial settings. Training the network means adjusting these dials until the music sounds perfect.

In [2]:
x = np.random.randn(3,1)
y = np.array([0., 0., 0,])

model = tf.keras.Sequential([
    tf.keras.layers.Dense(1)
])
model.compile(
    loss = tf.keras.losses.mae,
    optimizer=tf.keras.optimizers.SGD(),
    metrics=['mae']
)
model.fit(x, y, epochs=5)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 856ms/step - loss: 0.4180 - mae: 0.4180
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.4144 - mae: 0.4144
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.4108 - mae: 0.4108
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.4072 - mae: 0.4072
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.4036 - mae: 0.4036


In [3]:
# weights of the model
model.get_weights()

[array([[0.8130666]], dtype=float32), array([-0.01666667], dtype=float32)]

# Loss Function

- Concept: 
A loss function $L(y, \hat{y})$ measures how far off our model's predictions ($\hat{y}$) are from the ground truth targets ($y$). The total loss averaged across all training samples is often called the Cost Function $J(\theta)$.

- Math Formula (Mean Squared Error for regression): 
$$J(\theta) = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

- Intuition: 
The loss function provides a single scalar numerical score representing "error." Lower is better. Optimization is the act of tweaking $\theta$ to make $J(\theta)$ as close to zero (or its global minimum) as possible.

In [4]:
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

# Derivative and Slope

- Concept: 
For a single variable function $f(x)$, the derivative $\frac{df}{dx}$ measures the rate of change of $f(x)$ with respect to $x$ at a specific point.
- Formula: 
$$\frac{df}{dx} = \lim_{\Delta x \to 0} \frac{f(x + \Delta x) - f(x)}{\Delta x}$$
- Intuition: 
The derivative gives us the slope of the tangent line:
    - If $\frac{df}{dx} > 0$: Increasing $x$ increases $f(x)$. To reduce $f(x)$, move $x$ to the left (decrease $x$).
    - If $\frac{df}{dx} < 0$: Increasing $x$ decreases $f(x)$. To reduce $f(x)$, move $x$ to the right (increase $x$).
    - If $\frac{df}{dx} = 0$: You are at a stationary point (flat surface, local min, max, or saddle point).

![alt text](attachements/0ce3672a-83aa-4fa8-8ca6-6f43f83876c0.gif)

# Partial Derivatives

- Concept: 
Deep learning models have millions of input dimensions. A partial derivative $\frac{\partial J}{\partial w_i}$ measures how the overall loss $J$ changes when we tweak only one weight $w_i$, keeping all other weights constant.

$$\frac{\partial J}{\partial w_j} = \lim_{\Delta w_j \to 0} \frac{J(w_1, \dots, w_j + \Delta w_j, \dots, b) - J(w_1, \dots, w_j, \dots, b)}{\Delta w_j}$$

- Math Example: 
Suppose $J(w_1, w_2) = 3w_1^2 + 5w_2^3$.
    - Partial w.r.t $w_1$: $\frac{\partial J}{\partial w_1} = 6w_1$
    - Partial w.r.t $w_2$: $\frac{\partial J}{\partial w_2} = 15w_2^2$
- Intuition: It isolates the responsibility of a single weight in causing the model's total error.

In [5]:
# suppose the x = w1 and y = w2

![alt text](attachements/partial_derivative_visualization.gif)

# Gradient ($\nabla_\theta J$)

- Concept: 
The gradient is the vector containing all partial derivatives of the loss function with respect to every parameter in $\theta$.
- Math Formula: 
$$\nabla_\theta J(\theta) = \left[ \frac{\partial J}{\partial \theta_1}, \frac{\partial J}{\partial \theta_2}, \dots, \frac{\partial J}{\partial \theta_d} \right]^T$$

- Intuition:
    1. The gradient vector points in the direction of steepest increase (uphill) on the loss surface.
    2. Therefore, the negative gradient ($-\nabla_\theta J$) points in the direction of steepest decrease (downhill).
    
- Code (TensorFlow Automatic Differentiation):

In [6]:
x = tf.Variable(tf.constant(3, dtype=tf.float16)) # 3.0
with tf.GradientTape() as tape:
    y = x**2 # Loss function J(x)

# compute the gradient dy/dx
grad = tape.gradient(y, x)
print("Gradient at x = 3: ", grad.numpy()) # type: ignore # 2\*3 + 5 = 11.0

Gradient at x = 3:  6.0


# Step by Step Hand Calculation of Gradient Descent

When $J(\theta)$ depends on multiple parameters $(w_1, w_2, \dots, w_d, b)$, a partial derivative evaluates the sensitivity of $J$ with respect to one specific parameter while holding all other parameters strictly constant17:

$$\frac{\partial J}{\partial w_j} = \lim_{\Delta w_j \to 0} \frac{J(w_1, \dots, w_j + \Delta w_j, \dots, b) - J(w_1, \dots, w_j, \dots, b)}{\Delta w_j}$$

partial derivative of the Mean Squared Error $J(w, b)$ with respect to weight $w_j$ using the Chain Rule:

- Mean Square Error formula
$$J(w,b) = 
\frac{1}{2N} \sum_{i=1}^{N}
\left(w^T x^{(i)} + b - y^{(i)}\right)^2
$$


- Paritally derivating with w 
$$\frac{\partial J}{\partial w_j} = \frac{\partial}{\partial w_j} \left[ \frac{1}{2N} \sum_{i=1}^{N} \left( w^T x^{(i)} + b - y^{(i)} \right)^2 \right]$$

- Applying the chain rule $\frac{d}{dw}(u^2) = 2u \cdot \frac{du}{dw}$:

$$\frac{\partial J}{\partial w_j} = \frac{1}{2N} \sum_{i=1}^{N} 2 \cdot \left( w^T x^{(i)} + b - y^{(i)} \right) \cdot \frac{\partial}{\partial w_j} \left( \sum_{k=1}^d w_k x_k^{(i)} + b - y^{(i)} \right)

- $$Since $\frac{\partial}{\partial w_j}(w_k x_k^{(i)}) = x_j^{(i)}$ when $k = j$ (and $0$ otherwise):
$$\frac{\partial J}{\partial w_j} = \frac{1}{N} \sum_{i=1}^{N} \left( \hat{y}^{(i)} - y^{(i)} \right) x_j^{(i)}$$
$$\frac{\partial J}{\partial w_j} = \frac{1}{N} \sum_{i=1}^{N} e^{(i)}  x_j^{(i)}$$

- Similarly, for the bias $b$:$$\frac{\partial J}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} \left( \hat{y}^{(i)} - y^{(i)} \right)$$
$$\frac{\partial J}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} e^{(i)} $$

-  **Gradient Vector ($\nabla_\theta J$)**

$$\nabla_\theta J(\theta) = \begin{bmatrix} \frac{\partial J}{\partial w_1} \\ \frac{\partial J}{\partial w_2} \\ \vdots \\ \frac{\partial J}{\partial w_d} \\ \frac{\partial J}{\partial b} \end{bmatrix} \in \mathbb{R}^{d+1}$$

### Dataset 

- Dataset ($N=2$ samples, 1 feature):
    - Sample 1: $x^{(1)} = 1.0, y^{(1)} = 3.0$
    - Sample 2: $x^{(2)} = 2.0, y^{(2)} = 5.0$ (True relationship: $y = 2x + 1$
    
- Initial Model Parameters: $w^{(0)} = 0.0, b^{(0)} = 0.0$
- Learning Rate: $\alpha = 0.1$

---

### Step 1: Prediction 

Prediction : 
$$\hat{y} = w x + b$$

$$\hat{y}^{(1)} = w^{(0)} x^{(1)} + b^{(0)} = (0.0)(1.0) + 0.0 = 0.0$$
$$\hat{y}^{(2)} = w^{(0)} x^{(2)} + b^{(0)} = (0.0)(2.0) + 0.0 = 0.0$$

### Step 2: Compute Residual Errors

$$Residual: e = \hat{y} - y$$
$$e^{(1)} = 0.0 - 3.0 = -3.0$$
$$e^{(2)} = 0.0 - 5.0 = -5.0$$


### Step 3: Calculate Cost Function $J(w,b)$

$$J(w,b) = 
\frac{1}{2N} \sum_{i=1}^{N}(\hat{y} - y)^2
$$

- For N = 2

$$J(w, b) = \frac{1}{2 \times 2} \left[ \left(\hat{y}^{(1)} - y^{(1)}\right)^2 + \left(\hat{y}^{(2)} - y^{(2)}\right)^2 \right]$$Replacing $\left(\hat{y}^{(i)} - y^{(i)}\right)$ with error notation $e^{(i)}$:$$J(w, b) = \frac{1}{2(2)} \left( (e^{(1)})^2 + (e^{(2)})^2 \right)$$Since $2 \times 2 = 4$, the fraction simplifies to $\frac{1}{4}$:$$J(w, b) = \frac{1}{4} \left( (e^{(1)})^2 + (e^{(2)})^2 \right)$$

$$= \frac{1}{4} (9.0 + 25.0) = \mathbf{8.50}$$

$$J(w,b) = 8.50$$


### Step 4. Compute the Partial Derivatives

$$\frac{\partial J}{\partial w} = \frac{1}{N} \sum_{i=1}^{2} e^{(i)} x^{(i)} = \frac{1}{2} \left( (-3.0)(1.0) + (-5.0)(2.0) \right) = \frac{1}{2} (-3.0 - 10.0) = \mathbf{-6.50}$$

$$\frac{\partial J}{\partial b} = \frac{1}{N} \sum_{i=1}^{2} e^{(i)} = \frac{1}{2} \left( -3.0 + (-5.0) \right) = \frac{1}{2} (-8.0) = \mathbf{-4.00}$$

### Step 5. Gradient Descent Parameter Update

Applying $\theta_{new} = \theta_{old} - \alpha \nabla_\theta J$:$$w^{(1)} = w^{(0)} - \alpha \frac{\partial J}{\partial w} = 0.0 - (0.1)(-6.50) = \mathbf{+0.65}$$ $$b^{(1)} = b^{(0)} - \alpha \frac{\partial J}{\partial b} = 0.0 - (0.1)(-4.00) = \mathbf{+0.40}$$

### Step 6: Verification (Check Loss After 1 Step)
Recalculating predictions with $w^{(1)} = 0.65, b^{(1)} = 0.40$: $$\hat{y}^{(1)} = (0.65)(1.0) + 0.40 = 1.05 \quad \text{(Error) } e^{(1)} = 1.05 - 3.0 = -1.95\text{()}$$ $$\hat{y}^{(2)} = (0.65)(2.0) + 0.40 = 1.70 \quad \text{(Error) } e^{(2)} = 1.70 - 5.0 = -3.30\text{()}$$

$$J(w^{(1)}, b^{(1)}) = \frac{1}{4} \left( (-1.95)^2 + (-3.30)^2 \right) = \frac{1}{4} (3.8025 + 10.89) = \mathbf{3.673}$$

Result: In a single update step, our loss dropped from 8.50 down to 3.673, and parameters moved toward $w=2.0, b=1.0$.

# Batch, Stochastic(SGD), and Mini-Batch

How many data points we use to calculate $\nabla_\theta J$ before taking a step defines the optimization regime3more_horiz:

- Full Batch GD:   [==================== Dataset (N samples) ====================] -> 1 Update / Epoch
- Pure SGD:        [Sample 1] -> Update 1 | [Sample 2] -> Update 2 | ...           -> N Updates / Epoch
- Mini-Batch GD:   [Batch 1 (B=64)] -> Update 1 | [Batch 2 (B=64)] -> Update 2 ... -> N/B Updates / Epoch

## 1. Batch Gradint Descent (Full Batch)

Batch Gradient Descent (often called **Full-Batch Gradient Descent**) is the most straightforward form of gradient descent. 

Let’s break down the formula you wrote:
$$\theta_{t+1} = \theta_t - \eta \cdot \frac{1}{N} \sum_{i=1}^{N} \nabla_\theta \ell\left(y^{(i)}, f(x^{(i)}; \theta_t)\right)$$



- Explanation of Each Term
    - **$(\theta_t)$** → Parameters (weights, biases) of the model at iteration \(t\).
    - **$(\eta)$** → Learning rate, a scalar that controls the step size.
    - **$(N)$** → Total number of training samples in the dataset.
    - **$(\ell(y^{(i)}, f(x^{(i)}; \theta_t)))$** → Loss function for the \(i\)-th sample, comparing true label $(y^{(i)})$ with prediction $(f(x^{(i)}; \theta_t))$.
    - **$(\nabla_\theta)$** → Gradient of the loss with respect to parameters $(\theta)$.
    - **$(\frac{1}{N} \sum_{i=1}^{N})$** → Average gradient across the entire dataset.

- Intuition
    - At each step, you compute the gradient of the loss **over all samples**.
    - This gives the **true gradient** of the empirical risk (average loss).
    - You then update parameters by moving in the opposite direction of the gradient (to minimize loss).

- ✅ Pros
    - Converges smoothly because it uses the exact gradient.
    - Guarantees descent toward a local minimum (assuming convex loss).

- ❌ Cons
    - Computationally expensive for large datasets (must process all \(N\) samples per update).
    - Slower in practice compared to **Stochastic Gradient Descent (SGD)** or **Mini-Batch Gradient Descent**, which approximate the gradient using fewer samples.

## 2. Pure Stochastic Gradient Descent (SGD)

- Computes the gradient using a single randomly chosen example $(x^{(i)}, y^{(i)})$ per iteration

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_\theta \ell\left(y^{(i)}, f(x^{(i)}; \theta_t)\right)$$

- **Key difference**: No summation, just the gradient from a single sample.

- **Pros**: Fast updates, can escape local minima due to noise.

- **Cons**: 
   - Very noisy updates, convergence path zig-zags.
   - High variance causes severe trajectory noise[10]. Hardware execution cannot leverage GPU vectorization or matrix multiplication capabilities

## 3. Mini-Batch Gradient Descent

- This is the compromise between full-batch and SGD. Instead of all \(N\) samples or just one, we use a **batch of size (m)** (where $(1 < m < N)$).


$$\theta_{t+1} = \theta_t - \eta \cdot \frac{1}{m} \sum_{i=1}^{m} \nabla_\theta \ell\left(y^{(i)}, f(x^{(i)}; \theta_t)\right)$$

- **Key difference:** Average gradient over a small batch of samples.
- **Pros:** 
    - More stable than SGD, faster than full-batch.
    - Efficiency with Large Datasets, feasible when the dataset is too large to fit into memory.
    - Batch size $B$ aligns with tensor cores and memory architecture, achieving peak floating-point computational throughput.
    - Regularization effect: The small randomness from batches acts like a regularizer, helping avoid overfitting and sometimes escaping poor local minima.

- **Cons:** 
    - Still an approximation, but balances speed and stability.
    - Non-Determinism: Random shuffling of batches introduces variability in results; runs may not be identical.
    - Slower convergence then the SGD.

## Comparison Table Batch, SGD and Mini-Batch

| Method                  | Update Rule | Gradient Source | Pros | Cons |
|--------------------------|-------------|-----------------|------|------|
| **Batch (Full)**         | $(\frac{1}{N}\sum_{i=1}^N \nabla_\theta \ell)$ | All samples | Exact gradient, smooth convergence | Very slow for large datasets |
| **SGD**                  | $(\nabla_\theta \ell(y^{(i)}, f(x^{(i)};\theta)))$ | Single sample | Fast, can escape local minima | Very noisy, unstable |
| **Mini-Batch**           | $(\frac{1}{m}\sum_{i=1}^m \nabla_\theta \ell)$ | Small batch | Balance of speed & stability | Still approximate |

---

💡 Think of it like this:
- **Batch** = “Careful but slow.”
- **SGD** = “Fast but jittery.”
- **Mini-Batch** = “Balanced and practical.”

## Code Implementation of Mini-Batch GD

In [7]:
import numpy as np

def mini_batch_gradient_descent(X, y, epochs=100, lr=0.01, batch_size = 2):
    N, d = X.shape

    # Parameter Initialization
    w = np.zeros((d, 1))
    b = 0.0

    loss_history = []

    for epoch in range(epochs):
        # Shuffle dataset every epochs
        indices = np.random.permutation(N)
        X_shuffle = X[indices]
        y_shuffle = y[indices]

        for i in range(0, N, batch_size):
            X_batch = X_shuffle[i:i+batch_size]
            y_batch = y_shuffle[i:i+batch_size]
            B = len(X_batch)

            # Step 1: Prediction
            y_hat = np.dot(X_batch, w) + b

            # Step 2: Residual calculation
            error = y_hat - y_batch

            # Step 3: computer the gradient
            dw = (1/B) * np.dot(X_batch.T, error)
            db = (1/B) * np.sum(error)

            # Step 4: Parameter update
            w -= lr * dw
            b -= lr * db

        # Compute overall epoch MSE Loss
        total_loss = np.mean((np.dot(X, w) + b - y)**2) / 2
        loss_history.append(total_loss)

    return w, b, loss_history

In [8]:
np.random.seed(42)
X_data = np.random.randn(100, 2) # rows, columns

# y = 2*x1 + 3*x2 + 1 + (noice)
y_data = 2 * (X_data[:, 0:1]) + 3 * (X_data[:, 1:2]) + 1.0 + np.random.randn(100, 1)*0.01 

w_fit, b_fit, _ = mini_batch_gradient_descent(X_data, y_data, epochs=100, lr=0.01, batch_size = 16)

print(f"Fitted Weights:\n{w_fit.ravel()}\nFitted Bias: {b_fit:.4f}")

Fitted Weights:
[1.98814696 2.99726498]
Fitted Bias: 0.9960


# Graph naming convention

- **Global Minimum:** The absolute lowest point on the entire graph. There is no other point with a lower value anywhere in the function.

- **Local Minimum:** The lowest point within a specific, limited region (a "neighborhood"). It represents the bottom of a specific valley, though deeper valleys might exist elsewhere on the graph.

- **Saddle Point:** A point where the graph is flat (zero slope) but is not a minimum or a maximum. It curves upward in one direction and downward in another, resembling the shape of a horse's saddle.

![Local_global_saddle_point](attachements/2debdfec-3d3b-4378-a5b5-4016432b3c39.jpg)

## Code Example

In [9]:
import numpy as np
import plotly.graph_objects as go

# Generate data
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)

# Create a 2D grid (dataset) from the 1D arrays
X, Y = np.meshgrid(x, y)

# Calculate Z to create the peaks, valleys, and saddles
Z = X**4 - 6*X**2 + 4*X + Y**2

# Create the base 3d surface
fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale="Viridis")])

# Add the critical points as a 3D scatter plot layered on top
fig.add_trace(go.Scatter3d(
    x=[-1.637, 1.484, 0.101], 
    y=[0, 0, 0], 
    z=[-11.33, 0, 2], # Calculated Z values
    mode='markers+text',
    text=['Global Min', 'Local Min', 'Saddle Point'],
    marker=dict(size=8, color=['red', 'blue', 'black']),
    textfont=dict(color=['red', 'blue', 'black'], size=12)

))

fig.update_layout(
    title='Plotly 3D Surface: Real-Time Rotation',
    width=1000,
    height=700
)
fig.show()

# SGD with Momentum

- Standard Stochastic Gradient Descent (SGD) updates model weights by taking steps in the direction of the negative gradient of the loss function. While effective, it struggles in areas where the loss surface is much steeper in one dimension than another—often called "ravines." In these ravines, standard SGD oscillates wildly across the slopes, making very slow progress toward the actual minimum.

- Momentum solves this by borrowing a concept from physics. Instead of relying solely on the current gradient to determine the next step, it accumulates a velocity vector based on past gradients. It acts like a ball rolling down a hill—building up speed in directions where the gradient consistently points the same way, while dampening movement in directions where the gradient constantly changes sign.

## The Mathematics

- In standard SGD, given weights $\theta$, learning rate $\eta$, and loss function $J$, the update rule is:
$$\theta_{t+1} = \theta_t - \eta \nabla J(\theta_t)$$

- SGD with Momentum introduces a velocity term $v$, which is a moving average of past gradients. The update rules become:
$$v_{t+1} = \gamma v_t + \eta \nabla J(\theta_t)$$

$$\theta_{t+1} = \theta_t - v_{t+1}$$

- $\gamma$ (Momentum coefficient): Typically set to 0.9 or 0.99. 
    It determines how much of the previous velocity is retained. A value of 0 reduces the algorithm back to standard SGD.
- $\eta$(Learning rate): Scales the impact of the current gradient.

![alt text](attachements/sgd_with_momentum2.gif)

# Adaptive Learning Rate Optimizers

## 1. AdaGrad (Adaptive Gradient Algorithm)

The Problem with SGD & Momentum

Up until now, every parameter (whether it's weight #1 or weight #1,000,000) shared the exact same global learning rate $\eta$

- In real-world data, some features appear frequently (e.g., common words like "the"), while other features appear rarely (e.g., rare medical terms).

- With a fixed learning rate, rare features get updated very slowly because their gradients are zero most of the time.


### Example 

When a feature doesn't appear in a specific batch of training data, the model's prediction for that batch completely ignores that feature's weight. Because that weight had absolutely no impact on the error (loss) for that batch, **its gradient is exactly zero.**

Think about a text classification model where your inputs are words:

* **Frequent features:** A common word like "the" appears in almost every single batch. The gradient for its assigned weight is constantly active, meaning the weight is tweaked and updated at every step of SGD.
* **Rare features:** A rare medical term might only appear in 1 out of 1,000 batches. For the other 999 batches, the word isn't present, so its gradient is 0.

With standard SGD using a fixed learning rate ($\eta$), the weight update formula is:
$w = w - \eta \cdot \text{gradient}$

For the 999 batches where the rare word is missing, the math is simply $w = w - (\eta \cdot 0)$. The weight is completely frozen.

When that rare word finally shows up in the 1,000th batch, it gets a non-zero gradient. But because the learning rate $\eta$ is fixed globally, it only takes *one standard-sized step*. Meanwhile, the weights for frequent words have taken 1,000 steps and tuned themselves deeply to the data. The rare feature is left entirely under-optimized.

### Solution

This is exactly what AdaGrad solves. 

It tracks the historical gradients for every single weight. 

When a rare feature finally does get a non-zero gradient, AdaGrad gives it a massive, dynamically boosted learning rate so it can take a large step and "catch up" to the frequently updated weights.

- Frequently updated parameters $\rightarrow$ Learning rate shrinks (take smaller, cautious steps).
- Infrequently updated parameters $\rightarrow$ Learning rate stays large (take larger steps when the feature finally appears)


### The Math Formula

At time step $t$, for each parameter dimension $i$:

1. Accumulate past squared gradients: $$G_{t, i} = G_{t-1, i} + g_{t, i}^2$$ (where $g_{t, i} = \frac{\partial J}{\partial \theta_i}$ is the current gradient).

2. Update Parameters with Adaptive Learning Rate: $$\theta_{t+1, i} = \theta_{t, i} - \frac{\eta}{\sqrt{G_{t, i}} + \epsilon} \cdot g_{t, i}$$


### Fatal Flaw of AdaGrad

Look closely at $G_{t, i} = G_{t-1, i} + g_{t, i}^2$. Since $g_{t, i}^2 \ge 0$, $G_{t, i}$ strictly increases with every single training step.

- As $G_{t, i} \to \infty$, the effective learning rate $\frac{\eta}{\sqrt{G_{t, i}}}$ shrinks down to zero.

- **Result**: Late in training, the model completely stops learning long before it reaches the global minimum.

## 2. RMSProp (Root Mean Square Propagation)

Geoff Hinton invented RMSProp specifically to stop $G_{t, i}$ from growing to infinity. 

Instead of summing all past squared gradients from step 1 to infinity, RMSProp uses an Exponentially Decaying Moving Average (it "forgets" old gradients)

### The Math Formula

Update Moving Average of Squared Gradients ($v_t$): $$v_{t, i} = \beta \cdot v_{t-1, i} + (1 - \beta) \cdot g_{t, i}^2$$ (where $\beta$ is the decay factor, typically $\beta = 0.9$).Update Parameters: $$\theta_{t+1, i} = \theta_{t, i} - \frac{\eta}{\sqrt{v_{t, i}} + \epsilon} \cdot g_{t, i}$$

### Why RMSProp Works on 3D Landscapes
- On an asymmetric loss surface:Along steep directions (large gradients $g_t$), $v_t$ becomes large, so $\frac{\eta}{\sqrt{v_t}}$ becomes small $\rightarrow$ dampens oscillations.

- Along flat directions (small gradients $g_t$), $v_t$ becomes small, so $\frac{\eta}{\sqrt{v_t}}$ stays large $\rightarrow$ accelerates progress

### ✅ Pros of RMSProp
- Adaptive learning rate: It scales the learning rate for each parameter based on recent gradient magnitudes, making training stable even when gradients vary widely.

- Faster convergence: Handles non‑stationary objectives well (like deep networks) by adjusting step sizes dynamically.

- Less oscillation: Reduces zigzagging in steep regions compared to vanilla SGD.

- Works well for RNNs and deep architectures: Especially effective where gradients can explode or vanish.

### ⚠️ Cons of RMSProp
- Sensitive to hyperparameters: The decay rate (usually 0.9) and learning rate still need tuning.

- Can forget long‑term trends: Since it focuses on recent gradients, it may ignore older but relevant information.

- Noisy updates: In some cases, it can overshoot minima if the moving average isn’t well‑balanced.

- Not ideal for sparse gradients: AdaGrad performs better when gradients are sparse.

# Modern Standard - Adam & AdamW

## 1. Adam (Adaptive Moment Estimation)

### Why Adam Was Invented

Before Adam, researchers had two main breakthroughs:

1. **Momentum**: Accumulated an exponentially decaying average of **past gradients** (the 1st moment) to build velocity and push through flat regions.
2. **RMSProp**: Accumulated an exponentially decaying average of **past squared gradients** (the 2nd moment) to scale learning rates adaptively per coordinate.

Kingma &amp; Ba (2015) combined both ideas into a single unified framework called **Adaptive Moment Estimation (Adam)**

### Math Behind

At each time step $t$ with gradient $g_t = \nabla_\theta J(\theta_t)$:

1. **First Moment Estimate (Mean of Gradients)**
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
- 
    - $\beta_1 \in [0, 1)$ controls exponential decay (default $\beta_1 = 0.9$).
    - Represents the momentum / directional velocity

2. **Second Moment Estimate (Uncentered Variance of Gradients)**
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
- 
    - $\beta_1 \in [0, 1)$ controls exponential decay (default $\beta_1 = 0.9$).
    - Represents the adaptive step-scaling factor.

3. **Bias Correction (Crucial Step!)**

- Since $m_0 = \mathbf{0}$ and $v_0 = \mathbf{0}$ are initialized as zero vectors, both estimates are heavily biased toward zero during early iterations (especially when $\beta_1, \beta_2 \approx 1$).

- To correct this initial zero-bias, Adam computes corrected moments: 
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

4. **Parameter Update Rule**
$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$


### Numpy Implementation from scratch (Adam)

In [ ]:
import numpy as np

class AdamOptimizer:
    def __init__(self, lr=0.01, beta1= 0.9, beta2=0.99, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m = None
        self.v = None

    def update(self, w, dw):
        if self.m is None:
            self.m = np.zeros_like(w)
            self.v = np.zeros_like(w)

        self.t += 1

        # 1. Update biased 1st and 2nd moment estimates
        self.m = self.beta1 * self.m + (1 - self.beta1) * dw
        self.v = self.beta2 * self.v + (1 - self.beta2) * (dw**2)

        # 2. Compute bias-correction estimates
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)

        # 3. Update parameters
        w_updated = w - (self.lr / (np.sqrt(v_hat) + self.eps)) * m_hat
        return w_updated

In [11]:
# Suppose we want to minimize f(w) = w^2
w = np.array([5.0])   # initial weight
optimizer = AdamOptimizer(lr=0.1)

for step in range(50):
    dw = 2 * w   # gradient of f(w) = w^2
    w = optimizer.update(w, dw)
    print(f"Step {step+1}: w = {w[0]:.4f}")


Step 1: w = 4.9684
Step 2: w = 4.9368
Step 3: w = 4.9051
Step 4: w = 4.8736
Step 5: w = 4.8420
Step 6: w = 4.8104
Step 7: w = 4.7789
Step 8: w = 4.7473
Step 9: w = 4.7159
Step 10: w = 4.6844
Step 11: w = 4.6530
Step 12: w = 4.6216
Step 13: w = 4.5902
Step 14: w = 4.5589
Step 15: w = 4.5277
Step 16: w = 4.4965
Step 17: w = 4.4653
Step 18: w = 4.4342
Step 19: w = 4.4032
Step 20: w = 4.3722
Step 21: w = 4.3413
Step 22: w = 4.3104
Step 23: w = 4.2797
Step 24: w = 4.2489
Step 25: w = 4.2183
Step 26: w = 4.1877
Step 27: w = 4.1573
Step 28: w = 4.1268
Step 29: w = 4.0965
Step 30: w = 4.0663
Step 31: w = 4.0361
Step 32: w = 4.0060
Step 33: w = 3.9760
Step 34: w = 3.9461
Step 35: w = 3.9163
Step 36: w = 3.8866
Step 37: w = 3.8570
Step 38: w = 3.8274
Step 39: w = 3.7980
Step 40: w = 3.7686
Step 41: w = 3.7394
Step 42: w = 3.7102
Step 43: w = 3.6812
Step 44: w = 3.6522
Step 45: w = 3.6234
Step 46: w = 3.5946
Step 47: w = 3.5660
Step 48: w = 3.5374
Step 49: w = 3.5090
Step 50: w = 3.4806


## AdamW (Decoupled Weight Decay)

- **the assumption that L2 Regularization and Weight Decay are the exact same thing.**

- **In standard Stochastic Gradient Descent (SGD), they are mathematically identical. But in adaptive optimizers like Adam, treating them as the same thing severely degrades the model's ability to generalize.**
- **AdamW (Adam with decoupled Weight Decay) was introduced in 2017 to fix this specific flaw.**

    - Reason:
        1. SGD: $\theta_{t+1} = \theta_{t}-\eta * g_{t}$
            
            - $g_{t}=\nabla_{\theta} L (\theta_{t})$ : gradient of the loss at step 
𝑡
        2. Adam: 
        $$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
        $$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
        $$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$$
        $$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
        $$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t$$

        3. Why treating “SGD vs. mini‑batch” as the same hurts Adam

            - In **SGD**, the gradient estimate $(g_t)$ is noisy whether you use one sample or a mini‑batch. The update rule doesn’t change.  
            - In **Adam**, however, the variance of $(g_t)$ directly affects $(v_t)$ (the second moment).  
            - If you treat single‑sample SGD and mini‑batch SGD as “the same,” you ignore how batch size changes the variance of $(g_t)$.  
            - That variance feeds into $(\hat{v}_t)$, which controls the denominator in Adam’s update.  
            - Result: the effective learning rate per parameter changes drastically, and generalization can degrade.

$\theta_{t+1} = \theta_{t}-\eta * g_{t}$

### Problem with Adam

#### 1. **The Baseline: Weight Decay vs. L2 Regularization**

Both of these techniques exist to prevent overfitting by penalizing large weights. We want the model to keep its weights small and distributed.

- L2 Regularization: You add a penalty directly to the Loss Function.
$$Loss = \text{Original Loss} + \frac{\lambda}{2} \sum w^2$$

When you take the gradient of this new loss, you get:
    $$g_t = \nabla \text{Loss} + \lambda w_t$$

- Weight Decay: You ignore the loss function and instead directly shrink the weight by a small percentage during the Update Step.
$$w_{t+1} = w_t - \eta \cdot \lambda \cdot w_t - \eta \cdot g_t$$

In standard SGD, if you plug the L2 gradient into the SGD update rule, the math works out perfectly. L2 Regularization and Weight Decay behave exactly the same. Because of this, deep learning frameworks (like early PyTorch and TensorFlow) just implemented L2 Regularization inside Adam and called it "weight decay."

#### 2. **The Flaw in Standard Adam**

Adam is an adaptive optimizer. Just like AdaGrad (which scales learning rates based on historical gradients), Adam tracks the moving average of squared gradients and divides the learning rate by it.

If a weight has large, frequent gradients, Adam says: "I've updated this weight a lot, I'm going to shrink its effective learning rate so it doesn't bounce around too much."

If we use standard L2 Regularization with Adam, the penalty ($\lambda w_t$) is added to the gradient before Adam does its scaling.
$$\text{Update} = \frac{\eta}{\sqrt{v_t} + \epsilon} \cdot (g_t + \lambda w_t)$$

Notice what happens: the weight decay penalty ($\lambda w_t$) gets divided by the adaptive learning rate denominator ($\sqrt{v_t}$).

This creates a paradox:

1. A feature with huge gradients gets a large denominator ($\sqrt{v_t}$).
2. Adam shrinks the learning rate for that feature.
3. Because the L2 penalty is bundled inside, Adam also shrinks the penalty for that feature.
4. The weights that are moving the most—the exact weights most prone to overfitting—receive the least amount of regularization.

### The AdamW Fix: Decoupled Weight Decay

AdamW solves this by separating the two concepts. It completely removes the L2 penalty from the gradient calculation, letting Adam do its adaptive magic purely on the actual loss gradients.

Then, it applies true Weight Decay after the Adam update, directly to the weight:
$$w_{t+1} = w_t - \text{Adam\_Update}(g_t) - \eta \cdot \lambda \cdot w_t$$

By decoupling the weight decay from the gradient calculation, the decay rate ($\lambda$) applies equally to all weights, regardless of how frequent or rare their gradients are.

#### Why AdamW Took Over

- For a few years, researchers noticed that standard SGD with Momentum often achieved better generalization (lower test error) on image tasks than Adam, even though Adam trained much faster. 

- This led to a belief that "Adam overfits."Once AdamW was introduced and researchers realized the issue was just a bug in how L2 regularization interacted with adaptive scaling, AdamW quickly became the industry standard. 

- Today, it is the default optimizer for almost all state-of-the-art models, including Transformers (LLMs) and Vision Transformers.

# Interview Questions

## Q1: What is the main advantage of AdaGrad, and what is its primary limitation?
Answer: 
AdaGrad automatically adapts the learning rate per parameter, making larger updates for infrequent features. Its primary limitation is that $G_t$ accumulates squared gradients monotonically, causing the effective learning rate to decay down to zero prematurely and halt training.

## Q2: How does RMSProp modify AdaGrad to prevent learning rates from vanishing?
Answer: 
RMSProp replaces AdaGrad's cumulative sum of squared gradients with an exponentially decaying moving average ($v_t = \beta v_{t-1} + (1-\beta)g_t^2$) more_horiz. This restricts the window of past gradients, maintaining a stable step size throughout training.

## Q3. Why is bias correction ($\frac{1}{1-\beta^t}$) necessary in Adam?

Answer: 

At step $t=1$, with $m_0=0$ and $\beta_1=0.9$, $m_1 = 0.9(0) + 0.1 g_1 = 0.1 g_1$. 

Without correction, the update step would be artificially scaled down by 10x. 

Dividing by $(1 - 0.9^1) = 0.1$ corrects $m_1$ back to $1.0 g_1$. 

As $t \to \infty$, $\beta_1^t \to 0$, so the correction factor smoothly transitions to $1.0$.

## Q4: What is the key difference between Adam with $L_2$ regularization and AdamW?

Answer: 

In standard Adam, $L_2$ regularization gradients pass through the second-moment denominator $\sqrt{v_t}$, which scales down the decay for weights with large gradients. AdamW subtracts the weight decay term $\eta \lambda \theta_t$ directly from parameters, restoring true scale-invariant weight decay